# NestWatch nests in lidar: what does the habitat around a nest look like?

1. Pick one or more **species** (common name or 4-6 letter code) in the configuration cell.
2. The notebook keeps only nest attempts whose reported year is within **+/- 1 year** of a USGS 3DEP lidar (ALS) collection covering the nest (nearest-year product, as in `get_nearest_year_product` in [get_als.py](../../src/streaming/3dep/get_als.py)).
3. It then draws random matched nests and shows a **30 m radius 3D lidar cookie** centered on each one.

Tiles are streamed from the USGS 3DEP EPT on AWS, so the first draw of each cookie takes a few seconds. Cookies are cached on disk, so re-running is fast.

Run this notebook from `data_cleaning/data_exploration/` (the relative paths below assume that).

## Configuration

In [ ]:
SPECIES = ['Osprey','Great Horned Owl',"Bald Eagle"]   # list of common names ('Tree Swallow') and/or codes ('easblu'); see the species table below
N_SHOW = 8               # number of nests to draw
MAX_YEAR_DIFF = 1        # max |nest year - ALS collection year|
RADIUS = 30              # cookie radius in meters
ONE_PER_LOCATION = True  # at most one attempt per NestWatch Location ID (nest boxes are reused every year)
PER_SPECIES = False      # True: draw N_SHOW nests for each selected species; False: N_SHOW nests in total
SEED = None              # int for reproducible draws, None for different ones every run
HIGH_HAG = 3             # points at or above this height (m above ground) are always plotted in full
LOW_KEEP_FRAC = 0.1      # fraction of points below HIGH_HAG kept for plotting (1.0 = no thinning)
COLS = 4                 # cookies per row
COLOR_BY = 'intensity'  # 'height', 'intensity', or a false-color blend (see false_color.MODES)
HEIGHT_MODE = 'above_ground'  # 'above_ground' (terrain removed) or 'elevation' (raw Z, includes slope)
HEIGHT_CLIP = None       # color scale max in m; None = 99th percentile of shown cookies
CSV = '../../datasets/raw/NestWatch/NestWatch Open Dataset/attempts_locs_20260120.csv'
CACHE_DIR = '~/.cache/eagle_als_nestwatch'

In [ ]:
import hashlib
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.interpolate import LinearNDInterpolator, NearestNDInterpolator
from tqdm.auto import tqdm

sys.path.append('../../src/streaming/3dep/')
sys.path.append('../../scripts/3dep/')
from get_als import get_nearest_year_product
import false_color as fc
from dataset import load_als_cookie

rng = np.random.default_rng(SEED)
cache_dir = Path(CACHE_DIR).expanduser()
cache_dir.mkdir(parents=True, exist_ok=True)

## Load NestWatch attempts and choose species

The table below lists the most frequently reported species; any species in the data can be used.

In [ ]:
cols = ['Attempt ID', 'Location ID', 'Latitude', 'Longitude', 'Species Code', 'Species Name', 'Year', 'Subnational Code']
nw = pd.read_csv(CSV, usecols=cols, low_memory=False)
nw = nw.dropna(subset=['Latitude', 'Longitude', 'Year'])
nw = nw[(nw['Latitude'].between(-90, 90)) & (nw['Longitude'].between(-180, 180)) & ((nw['Latitude'] != 0) | (nw['Longitude'] != 0))]
nw['Year'] = nw['Year'].astype(int)
print(f"{len(nw):,} nest attempts, {nw['Species Name'].nunique()} species, years {nw['Year'].min()}-{nw['Year'].max()}")
counts = nw['Species Name'].value_counts().rename('attempts').to_frame().join(
    nw.drop_duplicates('Species Name').set_index('Species Name')['Species Code'])
counts.loc[SPECIES]

In [ ]:
wanted = {s.lower() for s in SPECIES}
sel = nw[nw['Species Name'].str.lower().isin(wanted) | nw['Species Code'].str.lower().isin(wanted)].copy()
unknown = wanted - set(sel['Species Name'].str.lower()) - set(sel['Species Code'].str.lower())
if unknown:
    raise ValueError(f"Not found in NestWatch: {sorted(unknown)}. Check the spelling against the table above.")
print(f"{len(sel):,} attempts for: {', '.join(sorted(sel['Species Name'].unique()))}")

## Match nests to lidar

For each unique (latitude, longitude, year), `get_nearest_year_product` finds the 3DEP product covering the point whose collection year is closest to the nest year. Nests with no coverage (or only products without a collection year), or whose nearest collection is more than `MAX_YEAR_DIFF` years away, are dropped.

In [ ]:
keys = sel[['Latitude', 'Longitude', 'Year']].drop_duplicates()
matches = []
for lat, lon, year in tqdm(keys.itertuples(index=False), total=len(keys), desc='matching to ALS'):
    try:
        m = get_nearest_year_product(lat, lon, year)
    except ValueError:   # every covering product lacks a collection year (all NaN)
        m = None
    matches.append(m if m is not None else {})
keys = pd.concat([keys.reset_index(drop=True), pd.DataFrame(matches)], axis=1)

nests = sel.merge(keys, on=['Latitude', 'Longitude', 'Year'], how='left')
has_als = nests['product_name_AWS'].notna()
ok = nests[has_als & (nests['year_diff_AWS'] <= MAX_YEAR_DIFF)].copy()
if ONE_PER_LOCATION:
    ok = ok.drop_duplicates('Location ID')

summary = pd.DataFrame({
    'attempts': nests.groupby('Species Name').size(),
    'with ALS coverage': nests[has_als].groupby('Species Name').size(),
    f'within +/-{MAX_YEAR_DIFF} yr': nests[has_als & (nests['year_diff_AWS'] <= MAX_YEAR_DIFF)].groupby('Species Name').size(),
    'selected pool': ok.groupby('Species Name').size(),
}).fillna(0).astype(int)
summary

In [ ]:
if ok.empty:
    raise ValueError('No nests match the lidar year criterion for this species selection.')
ok['year_diff_AWS'].value_counts().sort_index().rename('nests by |year diff|').to_frame()

## Cookie loading

`get_cookie` streams a cylinder of radius `RADIUS` around the nest, returns heights relative to the lowest point / ground, and caches the result. Nests whose cookie comes back empty or fails to stream are skipped and another nest is drawn.

In [ ]:
def height_above_ground(arr, max_ground=20000):
    """Z minus a ground surface interpolated from ASPRS class-2 points. NaN if there are too few ground points."""
    ground = arr[arr['Classification'] == 2]
    if len(ground) < 50:
        return np.full(len(arr), np.nan, dtype='float32')
    if len(ground) > max_ground:
        ground = ground[np.random.default_rng(0).choice(len(ground), max_ground, replace=False)]
    xy = np.column_stack([ground['X'], ground['Y']])
    q = np.column_stack([arr['X'], arr['Y']])
    surf = LinearNDInterpolator(xy, ground['Z'])(q)
    outside = np.isnan(surf)   # outside the ground-point hull: fall back to nearest ground point
    if outside.any():
        surf[outside] = NearestNDInterpolator(xy, ground['Z'])(q[outside])
    return (arr['Z'] - surf).astype('float32')


def get_cookie(row, min_points=50):
    """DataFrame[X, Y, Z, HAG, Intensity, ReturnType] (meters, relative to lowest point / ground) for a nest row, or None if too few points."""
    key = hashlib.md5(f"{row['Attempt ID']}_{row['product_name_AWS']}_{RADIUS}_v1".encode()).hexdigest()
    path = cache_dir / f"{key}.parquet"
    if path.exists():
        pts = pd.read_parquet(path)
        return pts if len(pts) >= min_points else None
    try:
        pipeline = load_als_cookie(
            als_tile_name=row['product_name_AWS'], lat=row['Latitude'], lon=row['Longitude'],
            point_crs='EPSG:4326', radius=RADIUS, out_crs=None,
        )
        arr = pipeline.arrays[0] if pipeline.arrays else []
    except Exception as e:  # network / EPT errors, point outside tile, ...
        warnings.warn(f"{row['Attempt ID']}: {type(e).__name__}: {str(e)[:100]}")
        return None  # not cached, may work on retry
    if len(arr):
        pts = pd.DataFrame({k: np.asarray(arr[k] - arr[k].min(), dtype='float32') for k in ['X', 'Y', 'Z']})
        pts['Intensity'] = np.asarray(arr['Intensity'], dtype='float32')
        pts['ReturnType'] = fc.return_type_codes(arr['ReturnNumber'], arr['NumberOfReturns'])
        pts['HAG'] = height_above_ground(arr)
    else:
        pts = pd.DataFrame(columns=['X', 'Y', 'Z', 'HAG', 'Intensity', 'ReturnType'], dtype='float32')
    pts.to_parquet(path)
    return pts if len(pts) >= min_points else None


def draw_cookies(pool, n):
    """Draw n loadable nests at random from `pool` (DataFrame). Returns [(row, cookie)]."""
    pool = pool.iloc[rng.permutation(len(pool))]
    out = []
    with tqdm(total=n, desc='loading cookies') as bar:
        for _, row in pool.iterrows():
            cookie = get_cookie(row)
            if cookie is not None:
                out.append((row, cookie))
                bar.update(1)
            if len(out) == n:
                break
    return out


def heights(tile):
    """Height values to plot (m). Above-ground heights are clipped at 0; cookies with no ground points fall back to raw elevation."""
    if HEIGHT_MODE == 'above_ground' and tile['HAG'].notna().all():
        return tile['HAG'].clip(lower=0)
    return tile['Z']


def point_colors(tile, height_max, color_by):
    if color_by == 'intensity':
        i = tile['Intensity']
        lo, hi = np.percentile(i, [2, 98])
        return dict(color=i.clip(lo, hi), colorscale='Cividis', cmin=lo, cmax=hi, showscale=False)
    if color_by in fc.MODES:
        rgb = fc.false_color(heights(tile), tile['Intensity'], tile['ReturnType'], mode=color_by, hag_max=height_max)
        return dict(color=fc.to_plotly_colors(rgb), showscale=False)
    return dict(color=heights(tile), colorscale='Viridis', cmin=0, cmax=height_max, showscale=True,
                colorbar=dict(title='height (m)', len=0.6))


def thin(tile):
    """Keep every point at or above HIGH_HAG; randomly keep only LOW_KEEP_FRAC of the (mostly ground) points below it."""
    high = (tile['HAG'] if tile['HAG'].notna().all() else tile['Z']) >= HIGH_HAG
    keep = high.to_numpy() | (rng.random(len(tile)) < LOW_KEEP_FRAC)
    return tile[keep]


def scene_grid(cookies, titles, height_max=None, width=330):
    """Grid of 3D scenes (COLS per row) sharing a height scale and axis ranges so cookies are comparable."""
    n = len(cookies)
    cols = min(COLS, n)
    rows = -(-n // cols)
    tiles = [thin(t) for t in cookies]
    height_max = height_max or HEIGHT_CLIP or max(np.percentile(heights(t), 99) for t in tiles)
    zmax = max(height_max, 2 * RADIUS * 0.4)
    fig = make_subplots(rows=rows, cols=cols, specs=[[{'type': 'scene'}] * cols for _ in range(rows)],
                        subplot_titles=titles, horizontal_spacing=0.01, vertical_spacing=0.08)
    for i, t in enumerate(tiles):
        r, c = divmod(i, cols)
        fig.add_trace(go.Scatter3d(
            x=t['X'], y=t['Y'], z=heights(t), mode='markers',
            marker=dict(size=1.2, **{**point_colors(t, height_max, COLOR_BY),
                                     'showscale': (i == n - 1 and COLOR_BY == 'height')}),
            hoverinfo='skip',
        ), row=r + 1, col=c + 1)
        fig.update_scenes(dict(
            xaxis=dict(range=[0, 2 * RADIUS], showticklabels=False, title=''),
            yaxis=dict(range=[0, 2 * RADIUS], showticklabels=False, title=''),
            zaxis=dict(range=[0, zmax], title='m'),
            aspectmode='manual', aspectratio=dict(x=1, y=1, z=zmax / (2 * RADIUS)),
        ), row=r + 1, col=c + 1)
    fig.update_annotations(font_size=11)
    fig.update_layout(width=width * cols + 60, height=340 * rows + 40, margin=dict(l=0, r=0, t=40, b=0), showlegend=False)
    return fig

## Nest cookies

Rotate the cookies with the mouse. The z-axis scale and the box are shared across all cookies, so vegetation height is directly comparable. The center of the cookie is the reported nest location (NestWatch coordinates are often the observer-entered location of the nest box, so expect some positional error). Titles give species, year, the lidar collection year, and state.

In [ ]:
if PER_SPECIES:
    drawn = [d for _, grp in ok.groupby('Species Name') for d in draw_cookies(grp, N_SHOW)]
else:
    drawn = draw_cookies(ok, N_SHOW)
if not drawn:
    raise RuntimeError('No cookies could be loaded; check network access to the 3DEP EPT.')

titles = [f"{r['Species Name']}<br>nest {r['Year']} / ALS {int(r['collection_year_AWS'])} - {r['Subnational Code']}" for r, _ in drawn]
scene_grid([c for _, c in drawn], titles).show()

In [ ]:
# attributes of the nests shown above, with a map link for each
shown = pd.DataFrame([r for r, _ in drawn])[['Attempt ID', 'Species Name', 'Year', 'collection_year_AWS', 'year_diff_AWS',
                                              'Subnational Code', 'Latitude', 'Longitude', 'product_name_AWS']].reset_index(drop=True)
shown['map'] = [f"https://www.google.com/maps/search/?api=1&query={a:.6f},{b:.6f}" for a, b in zip(shown['Latitude'], shown['Longitude'])]
shown